# Hito 4 — Escalamiento y PCA

**Proyecto Integrador — Minería de Datos 1**

## Propósito
En el Hito 3 encontramos que las tres variables numéricas del dataset (`age`,
`monthly_watch_time_mins`, `customer_support_tickets`) prácticamente no están correlacionadas
entre sí. Acá usamos PCA para confirmar esa evidencia de forma más rigurosa y para ver si,
aun así, es posible resumir la información en menos dimensiones sin perder demasiado.

## Variables utilizadas
Usamos las tres variables numéricas del dataset: `age`, `monthly_watch_time_mins` y
`customer_support_tickets`. No incluimos las variables categóricas (`subscription_plan`,
`country`, `favorite_genre`) porque PCA está pensado para variables numéricas continuas; a las
categóricas las usamos más adelante solo para colorear e interpretar los resultados, no como
entrada del PCA.


## 0. Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

sns.set_style("whitegrid")

df = pd.read_csv("../data/processed/streaming_users_clean.csv")
vars_num = ["age", "monthly_watch_time_mins", "customer_support_tickets"]
df[vars_num].describe()


## 1. Tratamiento de nulos para el PCA

**Contexto:** en el Hito 2 decidimos no imputar los nulos de estas columnas para no fabricar
información en la etapa de limpieza general. Pero PCA necesita datos completos para poder
calcularse.

**Decisión:** imputamos con la **mediana** únicamente para este análisis (no se modifica
`data/processed/streaming_users_clean.csv`). Usamos la mediana en vez del promedio porque es
menos sensible a valores extremos, y estas tres variables tienen algo de asimetría.


In [ ]:
print("Nulos antes de imputar:")
print(df[vars_num].isnull().sum())

imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(df[vars_num]), columns=vars_num)

print("\nNulos después de imputar:", X.isnull().sum().sum())


## 2. Escalamiento

**Por qué escalar:** `age` está en años (rango ~0-80), `monthly_watch_time_mins` está en minutos
(rango ~0-4.200) y `customer_support_tickets` está en cantidad de tickets (rango 0-5). Sin
escalar, la variable con los números más grandes (el consumo mensual) dominaría el PCA solo por
su magnitud, no porque sea más informativa. Usamos `StandardScaler`, que deja cada variable con
media 0 y desvío estándar 1.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=vars_num)
X_scaled.describe().round(2)


## 3. Aplicación de PCA

In [ ]:
pca = PCA()
componentes = pca.fit_transform(X_scaled)

varianza = pca.explained_variance_ratio_
varianza_acum = np.cumsum(varianza)

tabla_varianza = pd.DataFrame({
    "Componente": [f"PC{i+1}" for i in range(len(varianza))],
    "Varianza explicada": varianza.round(3),
    "Varianza acumulada": varianza_acum.round(3),
})
tabla_varianza


In [ ]:
plt.figure(figsize=(6,4))
plt.bar(tabla_varianza["Componente"], tabla_varianza["Varianza explicada"], color="slateblue")
plt.plot(tabla_varianza["Componente"], tabla_varianza["Varianza acumulada"], color="darkorange", marker="o")
plt.title("Varianza explicada por componente")
plt.ylabel("Proporción de varianza")
plt.show()


**Interpretación de la varianza explicada:** las tres componentes explican una proporción casi
idéntica de la varianza (≈33.6%, 33.4% y 33.0%). Esto **confirma con más rigor lo que ya
sospechábamos desde el Hito 3**: como las variables originales casi no están correlacionadas
entre sí, no existe una combinación de ellas que concentre mucha más información que las
variables por separado. En este dataset, PCA **no logra reducir la dimensionalidad de forma
útil**: para conservar el 90% de la información, necesitaríamos casi las 3 componentes igual.
Esto en sí mismo es un hallazgo válido, no una falla del análisis.

## 4. Interpretación de las componentes (cargas)

In [ ]:
cargas = pd.DataFrame(pca.components_.T, index=vars_num, columns=[f"PC{i+1}" for i in range(3)])
cargas.round(3)


In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(cargas, annot=True, cmap="coolwarm", center=0)
plt.title("Cargas de cada variable en cada componente")
plt.show()


**Interpretación:**
- **PC1** está dominada por `age` (0.73) y `customer_support_tickets` (0.67): es un eje que
  combina edad y soporte, sin relación clara con el consumo.
- **PC2** está dominada casi exclusivamente por `monthly_watch_time_mins` (0.89): es, en la
  práctica, "el eje del consumo".
- **PC3** es una combinación de las tres, con signos opuestos entre `age` y
  `customer_support_tickets`, y aporta la varianza restante.

Ninguna componente combina de forma clara las tres variables en un único "eje resumen"; cada una
queda dominada por una variable distinta, lo cual es coherente con que las variables originales ya
son casi independientes entre sí.


## 5. Visualización de usuarios en las primeras dos componentes

In [ ]:
df_pca = df.copy()
df_pca["PC1"] = componentes[:, 0]
df_pca["PC2"] = componentes[:, 1]

plt.figure(figsize=(7,5))
orden = ["Básico", "Estándar", "Premium"]
sns.scatterplot(data=df_pca, x="PC1", y="PC2", hue="subscription_plan", hue_order=orden,
                palette="viridis", alpha=0.4, s=20)
plt.title("Usuarios proyectados en PC1 y PC2, coloreados por plan")
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.legend(title="Plan")
plt.show()


In [ ]:
df_pca.groupby("subscription_plan")[["PC1","PC2"]].mean().round(3).loc[orden]


**Interpretación:** en el eje **PC2** (el "eje del consumo") se ve un ordenamiento claro:
el promedio pasa de -0.36 en Básico, a 0.15 en Estándar, a 0.55 en Premium. Esto es consistente
con el hallazgo del Hito 3: el consumo mensual es la variable que mejor separa a los usuarios
según su plan. En cambio, en **PC1** (edad + tickets) los tres planes están mezclados, sin
ningún orden aparente, porque ya sabíamos que ni la edad ni los tickets se relacionan con el plan.


## 6. Resumen del Hito 4

- Usamos las 3 variables numéricas del dataset (`age`, `monthly_watch_time_mins`,
  `customer_support_tickets`), imputando nulos solo para este análisis (mediana) y escalando con
  `StandardScaler`.
- Las 3 componentes explican una varianza casi idéntica (~33% cada una): **PCA no logra reducir
  la dimensionalidad de forma efectiva en este dataset**, porque las variables originales ya eran
  casi independientes (evidencia obtenida en el Hito 3).
- Aun así, PC2 resultó ser prácticamente equivalente a la variable de consumo mensual, y separa
  claramente a los usuarios por plan de suscripción, reforzando el hallazgo más importante del
  proyecto: **el consumo mensual es la variable que mejor diferencia a los usuarios según su plan**,
  mientras que la edad y los tickets de soporte aportan una dimensión adicional pero no relacionada
  con el plan.

Estos resultados van a ser la base de las conclusiones finales en `05_conclusiones.ipynb`.
